In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [13]:
import joblib
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, average_precision_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [8]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

In [9]:
results = []

In [10]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    pr_auc = average_precision_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    print(f"--- {name} ---")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | PR-AUC: {pr_auc:.4f}")
    print("Confusion Matrix:\n", cm)
    print()
    return {"model": name, "precision": precision, "recall": recall, "f1": f1, "pr_auc": pr_auc}

In [14]:
rf_baseline = joblib.load("baseline_rf.joblib")
results.append(evaluate_model("Random Forest (baseline)", rf_baseline, X_test, y_test))

--- Random Forest (baseline) ---
Precision: 0.9569 | Recall: 0.7817 | F1: 0.8605 | PR-AUC: 0.8031
Confusion Matrix:
 [[84971     5]
 [   31   111]]



In [15]:
xgb_plain = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric="aucpr"
)
xgb_plain.fit(X_train, y_train)
results.append(evaluate_model("XGBoost (plain)", xgb_plain, X_test, y_test))

--- XGBoost (plain) ---
Precision: 0.9619 | Recall: 0.7113 | F1: 0.8178 | PR-AUC: 0.8249
Confusion Matrix:
 [[84972     4]
 [   41   101]]



In [16]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print("scale_pos_weight:", round(spw, 1))

scale_pos_weight: 599.0


In [17]:
xgb_weighted = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=spw,
    random_state=42,
    n_jobs=-1,
    eval_metric="aucpr"
)
xgb_weighted.fit(X_train, y_train)
results.append(evaluate_model("XGBoost (scale_pos_weight)", xgb_weighted, X_test, y_test))

--- XGBoost (scale_pos_weight) ---
Precision: 0.9310 | Recall: 0.7606 | F1: 0.8372 | PR-AUC: 0.8137
Confusion Matrix:
 [[84968     8]
 [   34   108]]



In [18]:
lgbm_plain = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_plain.fit(X_train, y_train)
results.append(evaluate_model("LightGBM (plain)", lgbm_plain, X_test, y_test))

--- LightGBM (plain) ---
Precision: 0.0814 | Recall: 0.0493 | F1: 0.0614 | PR-AUC: 0.0056
Confusion Matrix:
 [[84897    79]
 [  135     7]]



In [19]:
lgbm_weighted = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    scale_pos_weight=spw,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_weighted.fit(X_train, y_train)
results.append(evaluate_model("LightGBM (scale_pos_weight)", lgbm_weighted, X_test, y_test))

--- LightGBM (scale_pos_weight) ---
Precision: 0.0279 | Recall: 0.8310 | F1: 0.0540 | PR-AUC: 0.0235
Confusion Matrix:
 [[80865  4111]
 [   24   118]]



In [20]:
comparison_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [21]:
comparison_df

,model,precision,recall,f1,pr_auc
0,XGBoost (plain),0.961905,0.711268,0.817814,0.824890
1,XGBoost (scale_pos_weight),0.931034,0.760563,0.837209,0.813679
2,Random Forest (baseline),0.956897,0.781690,0.860465,0.803127
3,LightGBM (scale_pos_weight),0.027903,0.830986,0.053992,0.023474
4,LightGBM (plain),0.081395,0.049296,0.061404,0.005598


In [22]:
proba_tr = lgbm_plain.predict_proba(X_train)[:, 1]
proba_te = lgbm_plain.predict_proba(X_test)[:, 1]

print("Train PR-AUC:", average_precision_score(y_train, proba_tr))
print("Test PR-AUC :", average_precision_score(y_test, proba_te))
print(pd.Series(proba_te).describe())

Train PR-AUC: 0.011959891697325354
Test PR-AUC : 0.00559848028009162
count    85118.00000
mean         0.00101
std          0.03177
min          0.00000
25%          0.00000
50%          0.00000
75%          0.00000
max          1.00000
dtype: float64


In [23]:
import lightgbm as lgb
print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


In [24]:
lgbm_dbg = LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42, n_jobs=-1)
lgbm_dbg.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    eval_metric="average_precision",
    callbacks=[lgb.log_evaluation(period=25)]
)

[25]	training's average_precision: 0.77318	training's binary_logloss: 0.0161416
[50]	training's average_precision: 0.69331	training's binary_logloss: 0.0458632
[75]	training's average_precision: 0.73241	training's binary_logloss: 0.0254653
[100]	training's average_precision: 0.618395	training's binary_logloss: 0.0380634
[125]	training's average_precision: 0.725854	training's binary_logloss: 0.0334658
[150]	training's average_precision: 0.284076	training's binary_logloss: 0.774207
[175]	training's average_precision: 0.400173	training's binary_logloss: 0.154527
[200]	training's average_precision: 0.451796	training's binary_logloss: 0.0609741


LGBMClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [25]:
lgbm_np = LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
lgbm_np.fit(X_train.values, y_train.values)
proba = lgbm_np.predict_proba(X_train.values)[:, 1]
print("Train PR-AUC (numpy input):", average_precision_score(y_train, proba))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train PR-AUC (numpy input): 0.011959891697325354


In [26]:
raw = lgbm_np.predict(X_train.values, raw_score=True)
print("raw score min/max:", raw.min(), raw.max())
print("PR-AUC on raw scores:", average_precision_score(y_train, raw))
print("Share of rows with proba exactly 0 or 1:", ((proba == 0) | (proba == 1)).mean())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


raw score min/max: -489478.355649107 68964.54386916463
PR-AUC on raw scores: 0.06417395049321835
Share of rows with proba exactly 0 or 1: 0.9998640538145492


In [27]:
lgbm_stable = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    reg_lambda=5.0,
    max_delta_step=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_stable.fit(X_train, y_train)
results.append(evaluate_model("LightGBM (stabilised)", lgbm_stable, X_test, y_test))

--- LightGBM (stabilised) ---
Precision: 0.9381 | Recall: 0.7465 | F1: 0.8314 | PR-AUC: 0.8161
Confusion Matrix:
 [[84969     7]
 [   36   106]]



In [28]:
comparison_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [29]:
comparison_df

,model,precision,recall,f1,pr_auc
0,XGBoost (plain),0.961905,0.711268,0.817814,0.824890
1,LightGBM (stabilised),0.938053,0.746479,0.831373,0.816054
2,XGBoost (scale_pos_weight),0.931034,0.760563,0.837209,0.813679
3,Random Forest (baseline),0.956897,0.781690,0.860465,0.803127
4,LightGBM (scale_pos_weight),0.027903,0.830986,0.053992,0.023474
5,LightGBM (plain),0.081395,0.049296,0.061404,0.005598


In [30]:
joblib.dump(xgb_plain, "xgb_plain.joblib")
joblib.dump(xgb_weighted, "xgb_weighted.joblib")
joblib.dump(lgbm_stable, "lgbm_stabilised.joblib")

['lgbm_stabilised.joblib']